# Skadi BI - Visualizações do Site

Notebook Python para gerar os gráficos dos dashboards de Chamados e Superadmin a partir das tabelas Gold.

Execute o notebook em um compute Python (CPU ou GPU), não em um SQL Warehouse.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

chamados = spark.table("gold_chamados")
usuarios = spark.table("gold_superadmin_usuarios")
rpa = spark.table("gold_superadmin_rpa")

chamados_pd = chamados.toPandas()
usuarios_pd = usuarios.toPandas()
rpa_pd = rpa.toPandas()

chamados_pd["data_hora_abertura"] = pd.to_datetime(chamados_pd["data_hora_abertura"])
chamados_pd["data_hora_encerramento"] = pd.to_datetime(chamados_pd["data_hora_encerramento"])

print(f"Chamados carregados: {len(chamados_pd)}")
print(f"Usuários carregados: {int(usuarios_pd['total_usuarios'].max()) if not usuarios_pd.empty else 0}")

## Cards de KPIs

In [ ]:
total_chamados = len(chamados_pd)
chamados_abertos = int(chamados_pd["esta_aberto"].sum())
chamados_finalizados = int(chamados_pd["esta_finalizado"].sum())
tempo_medio = chamados_pd.loc[
    chamados_pd["esta_finalizado"],
    "tempo_resolucao_minutos",
].mean()
total_usuarios = int(usuarios_pd["total_usuarios"].max()) if not usuarios_pd.empty else 0
ultima_carga_rpa = rpa_pd["ultima_execucao_sucesso"].max() if not rpa_pd.empty else None

kpis = pd.DataFrame({
    "KPI": [
        "Total de chamados",
        "Chamados abertos",
        "Chamados finalizados",
        "Tempo médio de resolução (min)",
        "Total de usuários",
        "Última carga RPA",
    ],
    "Valor": [
        total_chamados,
        chamados_abertos,
        chamados_finalizados,
        round(float(tempo_medio), 2) if pd.notna(tempo_medio) else None,
        total_usuarios,
        ultima_carga_rpa,
    ],
})
display(kpis)

## Dashboard de Chamados

In [ ]:
status = chamados_pd["status"].value_counts().sort_values(ascending=False)
plt.figure(figsize=(8, 5))
status.plot(kind="bar", color="#2563eb")
plt.title("Chamados por status")
plt.xlabel("Status")
plt.ylabel("Quantidade")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
display(plt.gcf())
plt.close()

In [ ]:
categoria = chamados_pd["categoria"].value_counts().sort_values(ascending=False)
plt.figure(figsize=(8, 5))
categoria.plot(kind="bar", color="#16a34a")
plt.title("Chamados por categoria")
plt.xlabel("Categoria")
plt.ylabel("Quantidade")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
display(plt.gcf())
plt.close()

In [ ]:
responsavel = chamados_pd["cod_usuario_responsavel"].fillna("NAO ATRIBUIDO").astype(str).value_counts()
plt.figure(figsize=(9, 5))
responsavel.plot(kind="bar", color="#9333ea")
plt.title("Chamados por responsável")
plt.xlabel("Código do responsável")
plt.ylabel("Quantidade")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
display(plt.gcf())
plt.close()

In [ ]:
aberturas = chamados_pd.assign(dia=chamados_pd["data_hora_abertura"].dt.date).groupby("dia").size().rename("abertos")
finalizacoes = chamados_pd.loc[chamados_pd["esta_finalizado"]].assign(dia=lambda df: df["data_hora_encerramento"].dt.date).groupby("dia").size().rename("finalizados")
evolucao = pd.concat([aberturas, finalizacoes], axis=1).fillna(0).sort_index()
plt.figure(figsize=(11, 5))
plt.plot(evolucao.index.astype(str), evolucao["abertos"], marker="o", label="Abertos")
plt.plot(evolucao.index.astype(str), evolucao["finalizados"], marker="o", label="Finalizados")
plt.title("Evolução diária de chamados")
plt.xlabel("Dia")
plt.ylabel("Quantidade")
plt.xticks(rotation=45, ha="right")
plt.legend()
plt.tight_layout()
display(plt.gcf())
plt.close()

In [ ]:
tempos = chamados_pd.loc[chamados_pd["esta_finalizado"], "tempo_resolucao_minutos"].dropna()
fig, eixos = plt.subplots(1, 2, figsize=(13, 5))
eixos[0].hist(tempos, bins=15, color="#f97316", edgecolor="white")
eixos[0].set_title("Histograma do tempo de resolução")
eixos[0].set_xlabel("Minutos")
eixos[0].set_ylabel("Quantidade")
eixos[1].boxplot(tempos, vert=True)
eixos[1].set_title("Boxplot do tempo de resolução")
eixos[1].set_ylabel("Minutos")
plt.tight_layout()
display(fig)
plt.close(fig)

## Dashboard de Superadmin

In [ ]:
usuarios_por_perfil = usuarios_pd.set_index("nivel_acesso")["quantidade_usuarios"].sort_values(ascending=False)
plt.figure(figsize=(8, 5))
usuarios_por_perfil.plot(kind="bar", color="#0891b2")
plt.title("Usuários por nível de acesso")
plt.xlabel("Nível de acesso")
plt.ylabel("Quantidade")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
display(plt.gcf())
plt.close()

In [ ]:
plt.figure(figsize=(8, 5))
status.plot(kind="bar", color="#dc2626")
plt.title("Chamados por status - visão Superadmin")
plt.xlabel("Status")
plt.ylabel("Quantidade")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
display(plt.gcf())
plt.close()

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(evolucao.index.astype(str), evolucao["abertos"], marker="o", label="Abertos")
plt.plot(evolucao.index.astype(str), evolucao["finalizados"], marker="o", label="Finalizados")
plt.title("Evolução diária de chamados - visão Superadmin")
plt.xlabel("Dia")
plt.ylabel("Quantidade")
plt.xticks(rotation=45, ha="right")
plt.legend()
plt.tight_layout()
display(plt.gcf())
plt.close()

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(13, 5))
eixos[0].hist(tempos, bins=15, color="#f97316", edgecolor="white")
eixos[0].set_title("Histograma do tempo de resolução - Superadmin")
eixos[0].set_xlabel("Minutos")
eixos[0].set_ylabel("Quantidade")
eixos[1].boxplot(tempos, vert=True)
eixos[1].set_title("Boxplot do tempo de resolução - Superadmin")
eixos[1].set_ylabel("Minutos")
plt.tight_layout()
display(fig)
plt.close(fig)